# Ingestion Pipeline — COSORA

Ingesta unificada: parser `python-docx` + tablas, chunking por frases, embeddings **MrBERT-es** o **E5** (configurable), ChromaDB en Drive o local.

**Prerequisito query:** ejecutar este notebook antes de `query_pipeline.ipynb`.

| Variable | Valores |
|----------|--------|
| `RUNTIME` | `auto` (detectado), `colab`, `local` |
| `EMBED_BACKEND` | `mrbert` \| `e5` |
| `INDEX_BOTH` | `True` indexa ambas colecciones (para benchmark) |

> **Nota:** Este notebook es auto-contenido. Funciona tanto en local como en Colab sin necesidad de subir archivos `.py` adicionales.

## 0. Setup y configuración

Instalación de dependencias e importaciones. La auto-detección `RUNTIME='auto'` detecta si estamos en Colab o en local.

In [1]:
%pip install -q python-docx chromadb sentence-transformers transformers torch rank_bm25 langchain-text-splitters

# Instalar antiword (Linux/Colab) o pywin32 (Windows) para .doc legacy
import sys, subprocess
if sys.platform.startswith("linux"):
    subprocess.run(["apt-get", "install", "-y", "antiword"], capture_output=True)
elif sys.platform == "win32":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pywin32"], check=False)

Note: you may need to restart the kernel to use updated packages.


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# AUTO-DETECT: Colab vs Local
# ═══════════════════════════════════════════════════════════════════════
import glob
import os
import re
import subprocess
import sys
import unicodedata
from pathlib import Path
from typing import Any

import chromadb
import torch
from docx import Document
from rank_bm25 import BM25Okapi
from transformers import AutoModel, AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

IN_COLAB = "google.colab" in sys.modules

# ─── Configuración ────────────────────────────────────────────────────
RUNTIME = "auto"           # "auto" | "colab" | "local"
INDEX_BOTH = True          # True → indexar mrbert + e5 (benchmark)
EMBED_BACKEND = "mrbert"   # "mrbert" | "e5" (solo si INDEX_BOTH=False)

if RUNTIME == "auto":
    RUNTIME = "colab" if IN_COLAB else "local"

if RUNTIME == "colab" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# ─── Rutas ────────────────────────────────────────────────────────────
def _find_project_root() -> Path:
    """Walk up from CWD until a dir containing .env or data/raw is found."""
    candidate = Path(".").resolve()
    for _ in range(6):
        if (candidate / ".env").exists() or (candidate / "data" / "raw").exists():
            return candidate
        parent = candidate.parent
        if parent == candidate:
            break
        candidate = parent
    return Path(".").resolve()

def resolve_paths(runtime, chroma_path_override=None):
    if runtime == "colab":
        docs_dir = "/content/drive/MyDrive/RAG_UPC_Final_project"
        chroma_path = chroma_path_override or f"{docs_dir}/chroma_db"
    else:
        root = _find_project_root()
        docs_dir = str(root / "data" / "raw")
        chroma_path = chroma_path_override or str(root / "notebooks" / "chroma_db")
    return docs_dir, chroma_path

DOCS_DIR, CHROMA_PATH = resolve_paths(RUNTIME)

print(f"IN_COLAB={IN_COLAB}  RUNTIME={RUNTIME}")
print(f"DOCS_DIR={DOCS_DIR}")
print(f"CHROMA_PATH={CHROMA_PATH}")
print(f"INDEX_BOTH={INDEX_BOTH}")
if not INDEX_BOTH:
    print(f"EMBED_BACKEND={EMBED_BACKEND}")

# ═══════════════════════════════════════════════════════════════════════
# COSORA SHARED — código embebido (no requiere cosora_shared.py)
# ═══════════════════════════════════════════════════════════════════════

EMBED_BACKENDS: dict[str, dict[str, Any]] = {
    "mrbert": {
        "model_id": "BSC-LT/MrBERT-es",
        "collection": "cosora_chunks_mrbert",
        "type": "transformers_cls",
        "query_prefix": "",
        "doc_prefix": "",
    },
    "e5": {
        "model_id": "intfloat/multilingual-e5-base",
        "collection": "cosora_chunks_e5",
        "type": "sentence_transformers",
        "query_prefix": "query: ",
        "doc_prefix": "passage: ",
    },
}


def backend_config(backend):
    if backend not in EMBED_BACKENDS:
        raise ValueError(f"EMBED_BACKEND desconocido: {backend}. Usa: {list(EMBED_BACKENDS)}")
    return EMBED_BACKENDS[backend]


def normalize_text(text):
    return re.sub(r"\s+", " ", text).strip()


def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    text = re.sub(r"[\u00ad\u2010-\u2015\u2212\uff0d]", "-", text)
    text = re.sub(r"[\u2018\u2019\u201a\u201b]", "'", text)
    text = re.sub(r"[\u201c\u201d\u201e\u201f]", '"', text)
    text = re.sub(r"[\u2022\u2023\u25cf\u25e6\u2043\u2219\u00b7]", "-", text)
    text = re.sub(r"(?m)^\s*\d{1,4}\s*$", "", text)
    text = re.sub(r"([.\-_=])\1{2,}", r"\1", text)
    text = re.sub(r"\s+([.,;:!?)])", r"\1", text)
    lines = [line for line in text.split("\n") if line.count("|") < 3]
    text = "\n".join(lines)
    text = re.sub(
        r"(?mi)^\s*(fecha|lugar|hora de inicio|hora de finalizacion|asistentes)\s*$",
        "",
        text,
    )
    text = re.sub(r"(?mi)^\s*(firmado|firma).*$", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def is_bad_chunk(chunk, min_words=40, min_alpha_ratio=0.65):
    words = len(chunk.split())
    if words < min_words:
        return True
    alpha_ratio = sum(c.isalpha() for c in chunk) / max(len(chunk), 1)
    return alpha_ratio < min_alpha_ratio


def chunk_text(text, chunk_size=500, overlap=100, min_chunk_size=50):
    if not text:
        return []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    raw_chunks = splitter.split_text(text)
    return [c for c in raw_chunks if len(c) >= min_chunk_size and not is_bad_chunk(c)]


def extract_docx(filepath):
    doc = Document(filepath)
    texts = []
    for paragraph in doc.paragraphs:
        text = normalize_text(paragraph.text)
        if text:
            texts.append(text)
    for table in doc.tables:
        for row in table.rows:
            row_texts = []
            seen = set()
            for cell in row.cells:
                text = normalize_text(cell.text)
                if not text or text in seen:
                    continue
                seen.add(text)
                row_texts.append(text)
            if row_texts:
                texts.append(" || ".join(row_texts))
    combined = "\n".join(texts).strip()
    return combined or None


def extract_doc(filepath):
    """Extract text from legacy .doc — antiword on Linux, win32com on Windows."""
    if sys.platform.startswith("linux"):
        result = subprocess.run(
            ["antiword", filepath],
            capture_output=True, text=True, check=False,
        )
        if result.returncode == 0 and result.stdout.strip():
            return result.stdout
        return None
    # Windows: use Word COM automation via pywin32
    try:
        import win32com.client
        word = win32com.client.Dispatch("Word.Application")
        word.Visible = False
        doc = word.Documents.Open(os.path.abspath(filepath))
        text = doc.Content.Text
        doc.Close(False)
        word.Quit()
        return text.strip() or None
    except Exception:
        pass
    # Fallback: try python-docx (works if .doc is OOXML-based)
    try:
        doc = Document(filepath)
        texts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        combined = "".join(texts)
        return combined or None
    except Exception:
        return None


def read_document(filepath):
    lower = filepath.lower()
    if lower.endswith(".docx"):
        return extract_docx(filepath)
    if lower.endswith(".doc"):
        return extract_doc(filepath)
    return None


def list_document_paths(docs_dir):
    paths = []
    paths.extend(glob.glob(os.path.join(docs_dir, "*.docx")))
    paths.extend(glob.glob(os.path.join(docs_dir, "*.doc")))
    return sorted(paths)


def process_documents(docs_dir):
    documents = []
    for filepath in list_document_paths(docs_dir):
        filename = os.path.basename(filepath)
        doc_id = os.path.splitext(filename)[0]
        raw = read_document(filepath)
        if not raw:
            continue
        cleaned = clean_text(raw)
        chunks = chunk_text(cleaned)
        if not chunks:
            continue
        documents.append({"doc_id": doc_id, "chunks": chunks, "filepath": filepath})
    return documents


class Embedder:
    def __init__(self, backend):
        self.backend = backend
        self.cfg = backend_config(backend)
        self._st_model = None
        self._model = None
        self._tokenizer = None

    def _load(self):
        if self.cfg["type"] == "sentence_transformers":
            if self._st_model is None:
                from sentence_transformers import SentenceTransformer
                self._st_model = SentenceTransformer(self.cfg["model_id"])
            return
        if self._model is None:
            self._tokenizer = AutoTokenizer.from_pretrained(self.cfg["model_id"])
            self._model = AutoModel.from_pretrained(self.cfg["model_id"])
            self._model.eval()

    def embed_one(self, text, *, is_query=False):
        self._load()
        if self.cfg["type"] == "sentence_transformers":
            prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
            vec = self._st_model.encode(prefix + text)
            return vec.tolist() if hasattr(vec, "tolist") else list(vec)
        assert self._tokenizer is not None and self._model is not None
        inputs = self._tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = self._model(**inputs)
        return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

    def embed_batch(self, texts, *, is_query=False, batch_size=16):
        self._load()
        if self.cfg["type"] == "sentence_transformers":
            prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
            prefixed = [prefix + t for t in texts]
            vecs = self._st_model.encode(prefixed, batch_size=batch_size, show_progress_bar=True)
            return vecs.tolist()
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            for text in batch:
                all_embeddings.append(self.embed_one(text, is_query=is_query))
        return all_embeddings


def index_documents(documents, backend, chroma_path, batch_size=32):
    cfg = backend_config(backend)
    embedder = Embedder(backend)
    all_texts, all_doc_ids, all_ids = [], [], []
    for doc in documents:
        for i, chunk in enumerate(doc["chunks"]):
            all_texts.append(chunk)
            all_doc_ids.append(doc["doc_id"])
            all_ids.append(f"{doc['doc_id']}__c{i:04d}")
    if not all_texts:
        return 0
    embeddings = embedder.embed_batch(all_texts, batch_size=batch_size)
    client = chromadb.PersistentClient(path=chroma_path)
    try:
        client.delete_collection(cfg["collection"])
    except Exception:
        pass
    collection = client.create_collection(
        name=cfg["collection"],
        metadata={"hnsw:space": "cosine", "embed_backend": backend},
    )
    metadatas = [
        {"doc_id": doc_id, "chunk_id": cid, "embed_backend": backend}
        for doc_id, cid in zip(all_doc_ids, all_ids)
    ]
    for i in range(0, len(all_texts), 5000):
        j = min(i + 5000, len(all_texts))
        collection.add(
            ids=all_ids[i:j],
            documents=all_texts[i:j],
            embeddings=embeddings[i:j],
            metadatas=metadatas[i:j],
        )
    return collection.count()


def get_chroma_collection(chroma_path, backend):
    cfg = backend_config(backend)
    client = chromadb.PersistentClient(path=chroma_path)
    return client.get_collection(cfg["collection"]), cfg


def dense_search(collection, embedder, query, k=100):
    q_vec = embedder.embed_one(query, is_query=True)
    res = collection.query(query_embeddings=[q_vec], n_results=k)
    return [
        {"text": doc, "meta": meta, "rank_dense": i}
        for i, (doc, meta) in enumerate(zip(res["documents"][0], res["metadatas"][0]))
    ]

print("\n✅ Código COSORA cargado correctamente")

c:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


IN_COLAB=False  RUNTIME=local
DOCS_DIR=C:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\data\raw
CHROMA_PATH=C:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\notebooks\chroma_db
INDEX_BOTH=True

✅ Código COSORA cargado correctamente


## 1. Procesar documentos (parser + chunking)

In [5]:
if not os.path.isdir(DOCS_DIR):
    raise FileNotFoundError(
        f"No existe DOCS_DIR: {DOCS_DIR}. "
        "Coloca los .docx/.doc en data/raw (local) o en Drive (colab)."
    )

documents = process_documents(DOCS_DIR)
total_chunks = sum(len(d["chunks"]) for d in documents)
print(f"{len(documents)} documentos, {total_chunks} chunks")
for doc in documents[:5]:
    print(f"  {doc['doc_id']}: {len(doc['chunks'])} chunks")


31 documentos, 799 chunks
  243591-DO-AVO-11-V07-251210: 4 chunks
  244170-DOB-AVO-05-V00-A0: 13 chunks
  244170-DOB-AVO-07-V00-A0: 36 chunks
  244170-DOB-AVO-09-V00: 18 chunks
  244170-DOB-AVO-10-V00-A0: 27 chunks


## 2. Indexar en ChromaDB

In [6]:
backends_to_index = list(EMBED_BACKENDS.keys()) if INDEX_BOTH else [EMBED_BACKEND]

for backend in backends_to_index:
    cfg = backend_config(backend)
    print(f"\n--- Indexando backend={backend} → colección {cfg['collection']} ---")
    n = index_documents(documents, backend, CHROMA_PATH, batch_size=16)
    print(f"ChromaDB: {n} chunks en {CHROMA_PATH}")


--- Indexando backend=mrbert → colección cosora_chunks_mrbert ---


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 5617.78it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB: 799 chunks en C:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\notebooks\chroma_db

--- Indexando backend=e5 → colección cosora_chunks_e5 ---


c:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\oriol\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 50/50 [00:19<00:00,  2.56it/s]


ChromaDB: 799 chunks en C:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\notebooks\chroma_db


## 3. Verificación

In [7]:
test_query = "¿Qué se decidió sobre el talud?"

backends_to_test = list(EMBED_BACKENDS.keys()) if INDEX_BOTH else [EMBED_BACKEND]

for backend in backends_to_test:
    collection, cfg = get_chroma_collection(CHROMA_PATH, backend)
    embedder = Embedder(backend)
    hits = dense_search(collection, embedder, test_query, k=3)

    print("\n" + "=" * 70)
    print(f"Backend: {backend}")
    print(f"Colección: {cfg['collection']} ({collection.count()} chunks)")
    print(f"Query prueba: {test_query}\n")
    for i, h in enumerate(hits, 1):
        print(f"{i}. [{h['meta']['doc_id']}] {h['text'][:120]}...")
print("\nOK: verificación completada")

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 10336.88it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Backend: mrbert
Colección: cosora_chunks_mrbert (799 chunks)
Query prueba: ¿Qué se decidió sobre el talud?

1. [254275-DO-AVO-17-V07] . La constructora informa que contactará con el fabricante y solicitarás las muestras de Deployé del proyecto. ANEJOS: 0...
2. [254275-DO-AVO-19-V07] . La constructora informa que contactará con el fabricante y solicitarás las muestras de Deployé del proyecto. ANEJOS: 0...
3. [254275-DO-AVO-22-V01] . 3.4. Protección frente al Radón || La DF indica que dado que la última propuesta de material de radón entregada por la...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13546.70it/s]



Backend: e5
Colección: cosora_chunks_e5 (799 chunks)
Query prueba: ¿Qué se decidió sobre el talud?

1. [254275-DO-AVO-14-V07] . AR-29 Estabilidad del talud || Posterior a la reunión de obra, DF envía en resultado de la verificación de la document...
2. [254275-DO-AVO-15-V07] . AR-29 Estabilidad del talud || Posterior a la reunión de obra, DF envía en resultado de la verificación de la document...
3. [254275-DO-AVO-16-V07] . AR-29 Estabilidad del talud || Posterior a la reunión de obra, DF envía en resultado de la verificación de la document...

OK: verificación completada
